# Phase 2 — Data Loading, Cleaning, EDA & Splitting
**Thesis:** Real-Time Fake News Detection Using BERT–BiLSTM  
**Student:** Mike Sayanka Lekaram | GC3183502  
**Dataset:** WELFake — 72,134 news articles (real + fake)

---
### What this notebook does
1. Mounts Google Drive and loads WELFake CSV
2. Cleans and preprocesses the text
3. Exploratory Data Analysis — 5 charts for your thesis
4. Splits into Train / Validation / Test sets and saves to Drive

> **Run every cell in order top to bottom using Shift+Enter**

---
## Cell 1 — Mount Google Drive
Run this first every session. It connects your permanent storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All thesis files live here
PROJECT_DIR = '/content/drive/MyDrive/FakeNewsThesis'
CHARTS_DIR  = f'{PROJECT_DIR}/charts'
DATA_DIR    = f'{PROJECT_DIR}/data'

os.makedirs(CHARTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR,   exist_ok=True)

print('Drive mounted.')
print('Project folder :', PROJECT_DIR)
print('Charts folder  :', CHARTS_DIR)
print('Data folder    :', DATA_DIR)

---
## Cell 2 — Install libraries
Colab already has most of these. This just makes sure everything is the right version.

In [ ]:
# Only wordcloud needs installing — everything else is pre-installed on Colab
!pip install wordcloud --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import string
import warnings

from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from collections import Counter

warnings.filterwarnings('ignore')

# Consistent chart style across all figures
plt.rcParams.update({
    'figure.dpi'      : 150,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 11,
    'xtick.labelsize' : 10,
    'ytick.labelsize' : 10,
    'font.family'     : 'DejaVu Sans',
})

REAL_COLOR = '#2E75B6'
FAKE_COLOR = '#C0504D'

print('All libraries imported successfully.')

---
## Cell 3 — Load WELFake CSV
Reads the CSV from your Drive and shows the first few rows so you can see what the data looks like.

In [ ]:
# ── Adjust this path if your CSV has a different name ──────────────────────────
CSV_PATH = f'{PROJECT_DIR}/WELFake_Dataset.csv'
# ───────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)

print('=== Dataset loaded ===')
print(f'Rows    : {len(df):,}')
print(f'Columns : {list(df.columns)}')
print()
print('First 3 rows:')
df.head(3)

---
## Cell 4 — Understand the label column
WELFake uses `label` column: **0 = Real news**, **1 = Fake news**

In [ ]:
print('Column names :', list(df.columns))
print()
print('Label value counts:')
print(df['label'].value_counts())
print()
print('Label meaning: 0 = Real  |  1 = Fake')
print(f'Real articles : {(df["label"]==0).sum():,}')
print(f'Fake articles : {(df["label"]==1).sum():,}')

---
## Cell 5 — Check for missing values and fix them
Missing text rows would crash the model later — we remove them now.

In [ ]:
print('=== Missing values before cleaning ===')
print(df.isnull().sum())
print()

rows_before = len(df)

# Drop rows where title OR text is missing
df.dropna(subset=['title', 'text'], inplace=True)

# Reset the index after dropping rows
df.reset_index(drop=True, inplace=True)

rows_after = len(df)
print(f'Rows removed  : {rows_before - rows_after:,}')
print(f'Rows remaining: {rows_after:,}')
print()
print('Missing values after cleaning:')
print(df.isnull().sum())

---
## Cell 6 — Combine title + text into one field
BERT reads one text input. We combine the headline and body into a single `content` column.

In [ ]:
# Combine title and body — separated by a period and space
df['content'] = df['title'].astype(str) + '. ' + df['text'].astype(str)

print('New column "content" created.')
print()
print('Example (first article):')
print(df['content'].iloc[0][:300], '...')

---
## Cell 7 — Text cleaning function
Removes noise from the text: URLs, HTML tags, punctuation, extra spaces.
This is standard NLP preprocessing — every word in the function is commented.

In [ ]:
def clean_text(text):
    """
    Clean a single news article string.
    Returns a lowercase string with noise removed.
    """
    # Convert to string in case of unexpected numeric values
    text = str(text)

    # Remove URLs  (http://... or https://...)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove HTML tags  (<b>, </p>, <div>, etc.)
    text = re.sub(r'<.*?>', '', text)

    # Remove content inside square brackets  [Reuters]
    text = re.sub(r'\[.*?\]', '', text)

    # Remove punctuation  (.,!?;: etc.)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Collapse multiple spaces/newlines into a single space
    text = re.sub(r'\s+', ' ', text).strip()

    # Convert to lowercase
    text = text.lower()

    return text


# Apply to all 72k articles — takes about 30–60 seconds
print('Cleaning text... (this takes about 30–60 seconds)')
df['content_clean'] = df['content'].apply(clean_text)

print('Done!')
print()
print('Before cleaning:')
print(df['content'].iloc[0][:200])
print()
print('After cleaning:')
print(df['content_clean'].iloc[0][:200])

---
## Cell 8 — Add article length column
Word count per article — used in EDA charts below.

In [ ]:
df['word_count'] = df['content_clean'].apply(lambda x: len(x.split()))

print('Word count stats:')
print(df.groupby('label')['word_count'].describe().round(1))
print()
print('Label 0 = Real  |  Label 1 = Fake')

---
## Cell 9 — EDA Chart 1: Class Distribution (Bar Chart)
Shows how many real vs fake articles are in the dataset. Goes into thesis Chapter 3.

In [ ]:
label_counts = df['label'].value_counts().sort_index()
labels       = ['Real (0)', 'Fake (1)']
colors       = [REAL_COLOR, FAKE_COLOR]
counts       = [label_counts[0], label_counts[1]]

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(labels, counts, color=colors, width=0.5, edgecolor='white', linewidth=0.8)

# Add count labels on top of each bar
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 300,
        f'{count:,}',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_title('Class Distribution — WELFake Dataset', fontweight='bold', pad=12)
ax.set_ylabel('Number of Articles')
ax.set_xlabel('Article Label')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, max(counts) * 1.15)

plt.tight_layout()
chart1_path = f'{CHARTS_DIR}/01_class_distribution.png'
plt.savefig(chart1_path, bbox_inches='tight')
plt.show()
print(f'Saved to: {chart1_path}')

---
## Cell 10 — EDA Chart 2: Article Length Distribution (Histogram)
Shows the spread of article word counts, separately for real and fake articles.

In [ ]:
real_wc = df[df['label'] == 0]['word_count']
fake_wc = df[df['label'] == 1]['word_count']

# Cap at 1500 words so the chart is readable (outliers distort the x-axis)
cap = 1500

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(real_wc[real_wc <= cap], bins=60, alpha=0.65,
        color=REAL_COLOR, label='Real news', edgecolor='white', linewidth=0.3)
ax.hist(fake_wc[fake_wc <= cap], bins=60, alpha=0.65,
        color=FAKE_COLOR, label='Fake news', edgecolor='white', linewidth=0.3)

ax.axvline(real_wc.median(), color=REAL_COLOR, linestyle='--',
           linewidth=1.5, label=f'Real median ({int(real_wc.median())} words)')
ax.axvline(fake_wc.median(), color=FAKE_COLOR, linestyle='--',
           linewidth=1.5, label=f'Fake median ({int(fake_wc.median())} words)')

ax.set_title('Article Length Distribution (words, capped at 1500)', fontweight='bold', pad=12)
ax.set_xlabel('Word Count')
ax.set_ylabel('Number of Articles')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
chart2_path = f'{CHARTS_DIR}/02_article_length_distribution.png'
plt.savefig(chart2_path, bbox_inches='tight')
plt.show()
print(f'Saved to: {chart2_path}')

---
## Cell 11 — EDA Chart 3: Top 20 Most Common Words (Bar Chart)
Shows which words appear most frequently across all articles.

In [ ]:
# Common English words (stopwords) that carry no meaning for fake news detection
STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had','do',
    'does','did','will','would','could','should','may','might','shall',
    'this','that','these','those','it','its','he','she','they','we','i',
    'you','his','her','their','our','my','your','said','says','also',
    'from','as','by','not','no','so','if','about','after','before','than',
    'more','all','one','two','three','new','us','can','up','out','into',
    'over','just','when','who','which','there','what','how','been','s','t'
}

# Count every word across all cleaned articles
all_words = ' '.join(df['content_clean']).split()
filtered  = [w for w in all_words if w not in STOPWORDS and len(w) > 2]
top20     = Counter(filtered).most_common(20)

words  = [w for w, _ in top20]
freqs  = [c for _, c in top20]

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.barh(words[::-1], freqs[::-1], color='#4472C4',
               edgecolor='white', linewidth=0.4)

ax.set_title('Top 20 Most Frequent Words — WELFake Dataset', fontweight='bold', pad=12)
ax.set_xlabel('Frequency')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
chart3_path = f'{CHARTS_DIR}/03_top20_words.png'
plt.savefig(chart3_path, bbox_inches='tight')
plt.show()
print(f'Saved to: {chart3_path}')

---
## Cell 12 — EDA Chart 4: Word Clouds (Real vs Fake)
Visual comparison of the most prominent words in real news vs fake news articles.

In [ ]:
real_text = ' '.join(df[df['label'] == 0]['content_clean'])
fake_text = ' '.join(df[df['label'] == 1]['content_clean'])

wc_config = dict(
    width=800, height=400,
    max_words=100,
    stopwords=STOPWORDS,
    background_color='white',
    collocations=False
)

wc_real = WordCloud(**wc_config, colormap='Blues').generate(real_text)
wc_fake = WordCloud(**wc_config, colormap='Reds' ).generate(fake_text)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(wc_real, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Real News — Most Common Words', fontweight='bold', pad=10, color=REAL_COLOR)

axes[1].imshow(wc_fake, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Fake News — Most Common Words', fontweight='bold', pad=10, color=FAKE_COLOR)

plt.tight_layout()
chart4_path = f'{CHARTS_DIR}/04_wordclouds_real_vs_fake.png'
plt.savefig(chart4_path, bbox_inches='tight')
plt.show()
print(f'Saved to: {chart4_path}')

---
## Cell 13 — EDA Chart 5: Average Word Count Real vs Fake (Comparison Bar)
Simple comparison bar chart showing whether fake or real articles tend to be longer.

In [ ]:
avg_wc = df.groupby('label')['word_count'].mean().round(1)

fig, ax = plt.subplots(figsize=(5, 4))

bars = ax.bar(
    ['Real News\n(label 0)', 'Fake News\n(label 1)'],
    [avg_wc[0], avg_wc[1]],
    color=[REAL_COLOR, FAKE_COLOR],
    width=0.45, edgecolor='white', linewidth=0.8
)

for bar, val in zip(bars, [avg_wc[0], avg_wc[1]]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f'{val:.0f} words',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_title('Average Article Length: Real vs Fake', fontweight='bold', pad=12)
ax.set_ylabel('Average Word Count')
ax.set_ylim(0, max(avg_wc[0], avg_wc[1]) * 1.2)

plt.tight_layout()
chart5_path = f'{CHARTS_DIR}/05_avg_word_count.png'
plt.savefig(chart5_path, bbox_inches='tight')
plt.show()
print(f'Saved to: {chart5_path}')

---
## Cell 14 — Split into Train / Validation / Test sets
Standard split: **80% train · 10% validation · 10% test**

- `stratify=y` ensures real/fake proportions are the same in all three sets
- `random_state=42` makes the split reproducible — you get the same split every time

In [ ]:
X = df['content_clean']   # the cleaned article text
y = df['label']            # 0 = real, 1 = fake

# First split: 80% train, 20% temp (which becomes val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: split the 20% temp into 50/50 → 10% val + 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print('=== Split complete ===')
print(f'Training set  : {len(X_train):>6,} rows  ({len(X_train)/len(df)*100:.0f}%)')
print(f'Validation set: {len(X_val):>6,} rows  ({len(X_val)/len(df)*100:.0f}%)')
print(f'Test set      : {len(X_test):>6,} rows  ({len(X_test)/len(df)*100:.0f}%)')
print()
print('Label distribution in training set:')
print(y_train.value_counts())

---
## Cell 15 — Save all three splits to Google Drive
These CSV files are saved permanently to your Drive. Phase 3 (BERT training) will load them directly.

In [ ]:
# Build DataFrames with both text and label
train_df = pd.DataFrame({'text': X_train, 'label': y_train}).reset_index(drop=True)
val_df   = pd.DataFrame({'text': X_val,   'label': y_val  }).reset_index(drop=True)
test_df  = pd.DataFrame({'text': X_test,  'label': y_test }).reset_index(drop=True)

# Save to Google Drive
train_path = f'{DATA_DIR}/train.csv'
val_path   = f'{DATA_DIR}/val.csv'
test_path  = f'{DATA_DIR}/test.csv'

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path,     index=False)
test_df.to_csv(test_path,   index=False)

print('=== Files saved to Google Drive ===')
print(f'train.csv  → {train_path}')
print(f'val.csv    → {val_path}')
print(f'test.csv   → {test_path}')
print()
print('These files are now permanently in your Drive.')
print('Phase 3 (BERT training) will load them from there.')

---
## Cell 16 — Final summary
Quick verification that everything ran correctly.

In [ ]:
print('============================================')
print('           PHASE 2 COMPLETE')
print('============================================')
print()
print(f'Total articles processed : {len(df):,}')
print(f'Real articles            : {(df["label"]==0).sum():,}')
print(f'Fake articles            : {(df["label"]==1).sum():,}')
print()
print('Splits saved:')
print(f'  train.csv  : {len(train_df):,} rows')
print(f'  val.csv    : {len(val_df):,} rows')
print(f'  test.csv   : {len(test_df):,} rows')
print()
print('Charts saved to Drive/FakeNewsThesis/charts/')
print('  01_class_distribution.png')
print('  02_article_length_distribution.png')
print('  03_top20_words.png')
print('  04_wordclouds_real_vs_fake.png')
print('  05_avg_word_count.png')
print()
print('Next step: Phase 3 — BERT + BiLSTM model training')